In [ ]:
"""
CS781 Phase 3 - Rhetorical Role Prediction (RR) with Model 2 + LoRMA
====================================================================

Base model : Model 2 (decoder-only, RoPE + GQA + SwiGLU, weight-tied, clean byte-level BPE 32k)
Method     : LoRMA (Low-Rank Multiplicative Adaptation, arXiv 2506.07621) on every attention / FFN projection
             (base weights frozen)
             + a small sentence-classification head (13 roles) trained from scratch
Task       : one label per sentence, scored with Macro-F1 on CL + IT test documents

How it works
------------
Model 2 is causal, so a sentence's representation can only use text to its LEFT.
We therefore feed the model a stretch of consecutive sentences (left context + a block of
"target" sentences), take the hidden state of the LAST token of every target sentence, and
classify it.  Every sentence is a target exactly once at inference time, and always has as
much left context as fits in the 1024-token window.

LoRMA (choose the variant with LORMA_VARIANT at the top of the config; details in section 3):
    h = M (W0 x), M = trainable d_out x d_out matrix built from two rank-r factors B (d_out x r) and A (r x d_out)
    "pi"   : M = I_pi(B A)  - row i of B A is rotated cyclically by i, which inflates the rank beyond r   (LoRMA_pi)
    "plus" : M = I + alpha/r * B A                                                                       (LoRMA+)
    Both start at M = I, so step 0 is exactly the pretrained Model 2.  Everything else (windows, head, class weights,
    oversampling, calibration, early stopping, seed) is identical to the LoRA file, so the runs are comparable.

Meant for a Kaggle notebook (paste into one cell, or run as a script).  Attach to the notebook:
    * the dataset with Model 2's checkpoint  (best.pt + best_meta.json from the pretraining run)
    * the dataset with the BPE tokenizer     (tokenizer.json + meta.json from bpe-final.ipynb)
The Phase 3 data is downloaded from the HF dataset repo (Kaggle secret named HF) unless found locally.

Imbalance handling (v2): per-domain capped inverse-sqrt class weights, oversampling of windows that contain rare labels,
domain-aware handling of labels unseen in a domain, and a 2-parameter post-hoc calibration (tau, unseen-label mask) picked on dev.

Outputs (WORK_DIR = phase3_rr_lorma_<variant>, so the two variants never overwrite each other):
    data_report.json            everything learned from the data (labels, counts, token stats, class weights)
    lorma_ckpt/best_adapter.pt  LoRMA + head weights of the best dev Macro-F1 (small, a few MB)
    lorma_ckpt/train_log.csv    per-epoch log
    prediction.json, submission.zip
"""
import os
import csv
import json
import math
import time
import zipfile
import random
from pathlib import Path
from dataclasses import dataclass
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================================
# CONFIG
# ============================================================================
SEED = 1234
LORMA_VARIANT = "pi"         # "pi" = LoRMA_pi (cyclic-rotation rank inflation) | "plus" = LoRMA+   (see section 3)
assert LORMA_VARIANT in ("pi", "plus"), LORMA_VARIANT

IS_KAGGLE = os.path.isdir("/kaggle/working")
WORK_DIR = f"/kaggle/working/phase3_rr_lorma_{LORMA_VARIANT}" if IS_KAGGLE else f"./phase3_rr_lorma_{LORMA_VARIANT}"
CKPT_DIR = os.path.join(WORK_DIR, "lorma_ckpt")
INPUT_ROOT = "/kaggle/input"

# ---- data (same repo / file names as phase3.ipynb) ----
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE3_FILES = {
    "CL_train": "Phase_3_RR/Train/CL_train.json",
    "CL_dev": "Phase_3_RR/Train/CL_dev.json",
    "IT_train": "Phase_3_RR/Train/IT_train.json",
    "IT_dev": "Phase_3_RR/Train/IT_dev.json",
    "label_vocab": "Phase_3_RR/Train/label_vocab.json",
    "CL_test": "Phase_3_RR/Test/CL_test.json",
    "IT_test": "Phase_3_RR/Test/IT_test.json",
}
LOCAL_PHASE3_DIR = os.environ.get("PHASE3_DIR", "")      # optional: root that contains Phase_3_RR/

# ---- base model (Model 2) ----
MODEL2_TAG = "decoder_only_rope_gqa_swiglu_clean_bpe32k_v1"   # tag written by the pretraining notebook
MODEL2_CKPT_PATH = ""        # set to an explicit best.pt / latest.pt to skip the automatic search
TOKENIZER_JSON = ""          # set to an explicit tokenizer.json to skip the automatic search
EXPECTED_VOCAB = 32_000

# ---- windowing (tokens) ----
MAX_LEN = 1024               # Model 2 context length
MAX_SENT_TOK = 160           # a sentence is cut to its first 160 tokens (p99 is well below this)
SENT_PREFIX = " "            # prepended to every sentence before BPE, so words look like "mid-text" words
TARGET_TOK = 320             # tokens of target sentences per window at inference
TRAIN_TARGET_RANGE = (160, 480)   # random target block size per window while training (augmentation)

# ---- LoRMA: Low-Rank Multiplicative Adaptation (arXiv 2506.07621) ----
# W0 stays frozen and its OUTPUT is multiplied by a trainable d_out x d_out matrix M built from rank-r factors:  h = M (W0 x)
#   pi   : M = I_pi(B A)            (I_pi rotates row i of B A cyclically by i -> rank inflation)
#   plus : M = I + (alpha / r) B A
# both start at M = I, i.e. exactly the pretrained model
LORMA_R = 16
LORMA_ALPHA = 32             # "plus" only: scaling = alpha / r   (the pi variant is unscaled, as in the paper)
LORMA_DROPOUT = 0.05         # "plus" only: dropout on the input of A   (the pi variant has none)
LORMA_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "w_gate", "w_up", "w_down")
TRAIN_FINAL_NORM = True      # also tune the 640 RMSNorm weights before the head

# ---- head ----
POOLING = "last"             # "last" = hidden of last token | "last_mean" = concat(last, mean over sentence tokens)
USE_POS_FEATURE = True       # relative position of the sentence in its document (bucketed)
N_POS_BUCKETS = 20
USE_DOMAIN_FEATURE = True    # CL / IT embedding
HEAD_HIDDEN = 256
HEAD_DROPOUT = 0.2

# ---- optimisation ----
FT_DROPOUT = 0.1             # residual dropout of Model 2 while fine-tuning (pretraining used 0.2)
EPOCHS = 12
MICRO_BS = 4
ACCUM = 2
LR_LORMA = 3e-4
LR_HEAD = 1e-3
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06
GRAD_CLIP = 1.0
# ---- class imbalance handling ----
DOMAINS = ["CL", "IT"]
CLASS_WEIGHT_POWER = 0.5     # weight_c = (N_d / (K_d * n_dc)) ** power : 0 = unweighted, 0.5 = inverse-sqrt, 1 = fully balanced
CLASS_WEIGHT_CLIP = (0.25, 8.0)   # cap: a label with 3 examples must not get a 100x gradient
PER_DOMAIN_WEIGHTS = True    # weights from each domain's OWN label counts (a label rare in IT but common in CL is up-weighted in IT only)
OVERSAMPLE_RARE = True       # repeat training windows that contain a rare label (rare = < RARE_FRAC of its domain's train sentences)
RARE_FRAC = 0.01
RARE_REPEAT = 3              # a window with a rare label is seen this many times per epoch (fresh dropout each time)
POSTHOC_CALIBRATION = True   # on dev: pick logit-adjustment strength tau and whether to mask labels unseen in a domain
TAU_GRID = (0.0, 0.25, 0.5, 0.75, 1.0)
LABEL_SMOOTHING = 0.0
PATIENCE = 4                 # stop when dev Macro-F1 has not improved for this many epochs
RUN_PREDICTION = True

AMP = {"enabled": False, "dtype": torch.float16}


def amp_ctx():
    return torch.autocast(device_type="cuda", dtype=AMP["dtype"], enabled=AMP["enabled"])


def log(*a):
    print(*a, flush=True)


def set_seed(s=SEED):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


# ============================================================================
# 1. DATA LOADING  (same layout handling as phase3.ipynb, token via Kaggle secret - never hard-coded)
# ============================================================================
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF")
        if tok:
            return tok
    except Exception:
        pass
    return os.environ.get("HF_TOKEN") or None


def resolve_phase3_files():
    paths = {}
    roots = []
    if LOCAL_PHASE3_DIR and Path(LOCAL_PHASE3_DIR).is_dir():
        roots.append(Path(LOCAL_PHASE3_DIR))
    roots += [Path(INPUT_ROOT), Path(WORK_DIR), Path.cwd()]
    wanted = {Path(p).name: k for k, p in PHASE3_FILES.items()}
    for root in roots:
        if not root.exists():
            continue
        for fp in root.rglob("*.json"):
            if fp.name in wanted:
                paths.setdefault(wanted[fp.name], str(fp))
    missing = [k for k in PHASE3_FILES if k not in paths]
    if missing:
        from huggingface_hub import hf_hub_download
        token = get_hf_token()
        for k in missing:
            log("downloading", PHASE3_FILES[k])
            paths[k] = hf_hub_download(repo_id=HF_DATASET_REPO, filename=PHASE3_FILES[k],
                                       repo_type="dataset", token=token)
    return paths


def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)



def normalize_documents(raw, split_name):
    """Normalize Phase 3 RR JSON into:
       [{"id": ..., "text": [...], "labels": [...] or None}]
    """
    docs = []

    def is_sentence_list(value):
        return (
            isinstance(value, list)
            and all(isinstance(s, str) for s in value)
        )

    def visit(obj, inherited_id=None):
        if isinstance(obj, list):
            # A list of strings may be a document's sentence list.
            if is_sentence_list(obj):
                docs.append({
                    "id": str(inherited_id or f"{split_name}_{len(docs)}"),
                    "text": obj,
                    "labels": None,
                })
                return

            for item in obj:
                visit(item, inherited_id)
            return

        if not isinstance(obj, dict):
            # Ignore metadata strings and other scalar values.
            return

        text = obj.get("text", obj.get("sentences"))
        labels = obj.get("labels", obj.get("label"))

        # Some files store sentences under a document field.
        if text is None:
            for key in ("data", "documents", "docs", "items"):
                if key in obj:
                    visit(obj[key], inherited_id or obj.get("id"))
                    return

            # Wrapper dictionaries may contain documents under arbitrary keys.
            for key, value in obj.items():
                if isinstance(value, (dict, list)):
                    visit(value, str(key))
            return

        if isinstance(text, str):
            text = [text]

        if not isinstance(text, list) or not all(
            isinstance(s, str) for s in text
        ):
            raise TypeError(
                f"{split_name}: invalid text format for document "
                f"{obj.get('id', inherited_id)}"
            )

        # Normalize labels only when present and valid.
        if isinstance(labels, list):
            labels = [str(x) for x in labels]
        else:
            labels = None

        docs.append({
            "id": str(obj.get("id", inherited_id or
                              f"{split_name}_{len(docs)}")),
            "text": text,
            "labels": labels,
        })

    visit(raw)

    if not docs:
        raise ValueError(f"{split_name}: no documents found")

    # Check IDs and label alignment.
    ids = [d["id"] for d in docs]
    if len(ids) != len(set(ids)):
        raise ValueError(f"{split_name}: duplicate document IDs detected")

    for d in docs:
        if d["labels"] is not None and len(d["labels"]) != len(d["text"]):
            raise ValueError(
                f"{split_name}: label/text length mismatch in {d['id']}: "
                f"{len(d['labels'])} labels vs {len(d['text'])} sentences"
            )

    return docs



def load_labels(raw):
    if isinstance(raw, dict) and "labels" in raw:
        raw = raw["labels"]
    if isinstance(raw, dict):
        raw = [k for k, _ in sorted(raw.items(), key=lambda kv: kv[1])]
    labels = [str(x) for x in raw]
    assert len(labels) == len(set(labels))
    return labels


def load_phase3():
    paths = resolve_phase3_files()
    raw = {k: read_json(p) for k, p in paths.items()}
    labels = load_labels(raw["label_vocab"])
    data = {k: normalize_documents(raw[k], k) for k in raw if k != "label_vocab"}
    l2i = {l: i for i, l in enumerate(labels)}
    for name, docs in data.items():
        labelled = name.endswith("_train") or name.endswith("_dev")
        for d in docs:
            d["domain"] = name.split("_")[0]
            if labelled:
                assert d["labels"] is not None and len(d["labels"]) == len(d["text"]), (name, d["id"])
                bad = [x for x in d["labels"] if x not in l2i]
                assert not bad, f"{name}/{d['id']}: unknown labels {set(bad)}"
    return data, labels, l2i


# ============================================================================
# 2. MODEL 2 ARCHITECTURE  (parameter names identical to the pretraining notebook so best.pt loads strictly;
#    the KV-cache generation path is dropped because classification does not need it)
# ============================================================================
@dataclass
class ModelConfig:
    vocab_size: int = 32_000
    d_model: int = 640
    n_layers: int = 8
    n_heads: int = 10
    n_kv_heads: int = 2
    ffn_hidden: int = 2048
    dropout: float = 0.2
    max_len: int = 1024 + 64
    pad_id: int = 0
    unk_id: int = 1
    eos_id: int = 2


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight).type_as(x)


def build_rope(head_dim, max_len, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x.float().chunk(2, dim=-1)
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)


def repeat_kv(x, groups):
    if groups == 1:
        return x
    B, KV, T, hd = x.shape
    return x[:, :, None].expand(B, KV, groups, T, hd).reshape(B, KV * groups, T, hd)


class GQAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.h = cfg.n_heads
        self.kv = cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.groups = self.h // self.kv
        self.q_proj = nn.Linear(cfg.d_model, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, cfg.d_model, bias=False)

    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        k, v = repeat_kv(k, self.groups), repeat_kv(v, self.groups)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w_gate = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_up = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_down = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn = GQAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn = SwiGLU(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin):
        x = x + self.drop(self.attn(self.norm1(x), cos, sin))
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = RMSNorm(cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        cos, sin = build_rope(cfg.d_model // cfg.n_heads, cfg.max_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def hidden(self, idx):
        """(B, T) token ids -> (B, T, d_model) final-norm hidden states (what the tied LM head would read)"""
        T = idx.size(1)
        cos, sin = self.rope_cos[:T][None, None], self.rope_sin[:T][None, None]
        x = self.drop(self.embed(idx))
        for blk in self.blocks:
            x = blk(x, cos, sin)
        return self.final_norm(x)


# ============================================================================
# 3. LoRMA  (Low-Rank Multiplicative Adaptation)
#    LoRA   adds a low-rank term to the weights:   W' = W0 + (alpha / r) B A
#    LoRMA  multiplies the weights by a matrix:    W' = M W0        (M is d_out x d_out, built from B A)
#      plus : M = I + (alpha / r) B A
#      pi   : M = I_pi(B A),  I_pi(X)[i, j] = X[i, (j - i) mod d]   (row i of X rotated cyclically to the right by i)
#             B A has rank <= r, but the row rotation "inflates" the rank of I_pi(B A) far beyond r
#    Both start at M = I (the adapted model is EXACTLY the pretrained one at step 0):
#      plus : B = 0
#      pi   : B[:, 0] = 1, A[0] = e_0 (the rank-1 matrix 1 e_0^T, which I_pi maps to I), all other columns of B = 0
#    W0 can be non-square (k/v: 640 -> 128, gate/up: 640 -> 2048); M always acts on the OUTPUT side (d_out x d_out).
#    The forward pass is  h = h0 + Delta h0  with  h0 = W0 x  and  Delta = M - I  (Delta is computed in fp32, so the
#    small learned part is not rounded away next to the identity when autocast runs the matmuls in bf16/fp16).
# ============================================================================
_PI_INDEX = {}


def pi_index(d, device):
    """idx[i, j] = (j - i) mod d.  Gathering row i of X with idx[i] rotates it to the right by i.  Shared by all layers of width d."""
    key = (d, str(device))
    if key not in _PI_INDEX:
        ar = torch.arange(d, device=device)
        _PI_INDEX[key] = (ar[None, :] - ar[:, None] + d) % d
    return _PI_INDEX[key]


def pi_rotate(X):
    """I_pi(X)[i, j] = X[i, (j - i) mod d] for a square matrix X"""
    return X.gather(1, pi_index(X.size(0), X.device))


class LoRMALinear(nn.Module):
    """h = M (W0 x), W0 frozen.  B: (d_out, r), A: (r, d_out).  See the section header for M."""

    def __init__(self, base: nn.Linear, variant: str, r: int, alpha: float, dropout: float):
        super().__init__()
        assert base.bias is None, "Model 2 uses bias-free projections"
        assert variant in ("plus", "pi"), variant
        d = base.out_features
        assert 1 <= r <= d, (r, d)
        self.base = base
        self.variant = variant
        self.r = r
        self.d = d
        self.scaling = alpha / r if variant == "plus" else 1.0
        A = torch.empty(r, d)
        nn.init.kaiming_uniform_(A, a=math.sqrt(5))
        B = torch.zeros(d, r)
        if variant == "pi":
            A[0] = 0.0
            A[0, 0] = 1.0                  # row 0 of A = e_0
            B[:, 0] = 1.0                  # column 0 of B = all ones   ->   B A = 1 e_0^T   ->   I_pi(B A) = I
        self.lorma_A = nn.Parameter(A)
        self.lorma_B = nn.Parameter(B)
        self.drop = nn.Dropout(dropout) if (dropout > 0 and variant == "plus") else nn.Identity()
        self.merged = False
        self.enabled = True                # False -> behave exactly like the frozen base layer (used by the self-check)
        self._w0 = None                    # backup of the original weight while merged

    @property
    def weight(self):                      # lets code that peeks at .weight keep working
        return self.base.weight

    def delta_matrix(self):
        """Delta = M - I as a (d, d) fp32 matrix (differentiable).  autocast is switched off so that B A, the rotation and
        the subtraction of the identity are done in fp32."""
        with torch.autocast(device_type=self.lorma_A.device.type, enabled=False):
            BA = self.lorma_B.float() @ self.lorma_A.float()
            if self.variant == "pi":
                eye = torch.eye(self.d, device=BA.device, dtype=BA.dtype)
                return pi_rotate(BA) - eye
            return BA * self.scaling

    def forward(self, x):
        h0 = self.base(x)
        if self.merged or not self.enabled:
            return h0
        if self.variant == "plus":         # Delta h0 = (alpha / r) B (A h0): never build the d x d matrix
            low = F.linear(F.linear(self.drop(h0), self.lorma_A.to(h0.dtype)), self.lorma_B.to(h0.dtype))
            return h0 + low * self.scaling
        return h0 + F.linear(h0, self.delta_matrix().to(h0.dtype))

    @torch.no_grad()
    def merge(self):
        """W' = M W0 = W0 + Delta W0.  M is not guaranteed to be invertible, so W0 is backed up for unmerge"""
        if not self.merged:
            w = self.base.weight
            self._w0 = w.detach().clone()
            with torch.autocast(device_type=w.device.type, enabled=False):
                w.add_((self.delta_matrix() @ self._w0.float()).to(w.dtype))
            self.merged = True

    @torch.no_grad()
    def unmerge(self):
        if self.merged:
            self.base.weight.copy_(self._w0)
            self._w0 = None
            self.merged = False


def inject_lorma(lm: DecoderLM, targets=LORMA_TARGETS, variant=LORMA_VARIANT, r=LORMA_R, alpha=LORMA_ALPHA,
                 dropout=LORMA_DROPOUT):
    """freeze everything in the LM, then wrap every Linear whose attribute name is in `targets`"""
    for p in lm.parameters():
        p.requires_grad = False
    n = 0
    for blk in lm.blocks:
        for parent in (blk.attn, blk.ffn):
            for name, child in list(parent.named_children()):
                if name in targets and isinstance(child, nn.Linear):
                    setattr(parent, name, LoRMALinear(child, variant, r, alpha, dropout))
                    n += 1
    assert n > 0, "no Linear layer matched LORMA_TARGETS"
    if TRAIN_FINAL_NORM:
        lm.final_norm.weight.requires_grad = True
    return n


def lorma_modules(model):
    return [m for m in model.modules() if isinstance(m, LoRMALinear)]


def merge_lorma(model):
    for m in lorma_modules(model):
        m.merge()


def unmerge_lorma(model):
    for m in lorma_modules(model):
        m.unmerge()


def set_lorma_enabled(model, flag):
    for m in lorma_modules(model):
        m.enabled = flag


# ============================================================================
# 4. TOKENIZER + DATA -> TOKEN WINDOWS
# ============================================================================
class LegalByteBPE:
    def __init__(self, tokenizer_json, meta_path=None):
        from tokenizers import Tokenizer
        self.tokenizer = Tokenizer.from_file(tokenizer_json)
        self.vocab_size = self.tokenizer.get_vocab_size()
        vocab = self.tokenizer.get_vocab()
        self.pad_id = vocab["<|pad|>"]
        self.unk_id = vocab["<|unk|>"]
        self.eos_id = vocab["<|endoftext|>"]
        if meta_path and os.path.isfile(meta_path):
            meta = read_json(meta_path)
            assert meta["vocab_size"] == self.vocab_size

    def encode(self, text):
        # drop pad/eos in case the tokenizer's post-processor appends them: pooling reads the LAST token of a sentence
        return [i for i in self.tokenizer.encode(text).ids if i not in (self.pad_id, self.eos_id)]


def load_tokenizer():
    cands = []
    if TOKENIZER_JSON:
        cands.append(os.path.dirname(TOKENIZER_JSON) or ".")
    for root in (INPUT_ROOT, os.getcwd(), WORK_DIR):
        if os.path.isdir(root):
            for d, _, files in os.walk(root):
                if "tokenizer.json" in files:
                    cands.append(d)
    for d in cands:
        path = TOKENIZER_JSON or os.path.join(d, "tokenizer.json")
        try:
            tok = LegalByteBPE(path, os.path.join(d, "meta.json"))
        except Exception as e:
            log(f"skip tokenizer {path}: {e}")
            continue
        if tok.vocab_size != EXPECTED_VOCAB:
            log(f"skip tokenizer {path}: vocab {tok.vocab_size} != {EXPECTED_VOCAB} (probably another model's BPE)")
            continue
        log(f"tokenizer: {path} | vocab={tok.vocab_size} pad={tok.pad_id} unk={tok.unk_id} eos={tok.eos_id}")
        return tok
    raise FileNotFoundError("no 32k tokenizer.json found - attach the bpe-final.ipynb output dataset")


def tokenize_documents(docs, tok):
    """per document: list of token-id lists, one per sentence, each cut to MAX_SENT_TOK (never empty)"""
    out = []
    for d in docs:
        sents = []
        for s in d["text"]:
            ids = tok.encode(SENT_PREFIX + s.strip())[:MAX_SENT_TOK]
            sents.append(ids if ids else [tok.unk_id])
        d["sent_ids"] = sents
        out.append(sents)
    return out


def make_windows(sent_ids, rng=None):
    """
    Cover EVERY sentence exactly once as a *target*.  Each window = [left-context sentences][target block],
    total tokens <= MAX_LEN.  Returns a list of (ctx_start, tgt_start, tgt_end) sentence ranges
    (ctx_start <= tgt_start < tgt_end, half-open).
    rng=None -> deterministic (inference / dev);  rng given -> random target-block size (training augmentation).
    """
    n = len(sent_ids)
    lens = [len(s) for s in sent_ids]
    wins, i = [], 0
    while i < n:
        budget = TARGET_TOK if rng is None else rng.randint(*TRAIN_TARGET_RANGE)
        budget = min(budget, MAX_LEN)
        j, used = i, 0
        while j < n and (j == i or used + lens[j] <= budget):
            used += lens[j]
            j += 1
        # left context: as many previous sentences as still fit
        c, room = i, MAX_LEN - used
        while c > 0 and lens[c - 1] <= room:
            c -= 1
            room -= lens[c]
        wins.append((c, i, j))
        i = j
    return wins


def window_example(doc, label_ids, win, domain_id):
    c, a, b = win
    ids, ends, starts = [], [], []
    for k in range(c, b):
        starts.append(len(ids))
        ids.extend(doc["sent_ids"][k])
        ends.append(len(ids) - 1)
    n = len(doc["sent_ids"])
    return {
        "ids": ids,
        "ends": ends[a - c:],
        "starts": starts[a - c:],
        "labels": [label_ids[k] for k in range(a, b)] if label_ids is not None else [-100] * (b - a),
        "pos": [min(int(k / n * N_POS_BUCKETS), N_POS_BUCKETS - 1) for k in range(a, b)],
        "domain": domain_id,
        "sent_idx": list(range(a, b)),
    }


def collate(examples, pad_id, device):
    B = len(examples)
    T = max(len(e["ids"]) for e in examples)
    S = max(len(e["ends"]) for e in examples)
    ids = torch.full((B, T), pad_id, dtype=torch.long)
    ends = torch.zeros(B, S, dtype=torch.long)
    starts = torch.zeros(B, S, dtype=torch.long)
    labels = torch.full((B, S), -100, dtype=torch.long)
    pos = torch.zeros(B, S, dtype=torch.long)
    smask = torch.zeros(B, S, dtype=torch.bool)
    dom = torch.zeros(B, dtype=torch.long)
    for r, e in enumerate(examples):
        ids[r, :len(e["ids"])] = torch.tensor(e["ids"])           # right padding is safe: attention is causal
        m = len(e["ends"])
        ends[r, :m] = torch.tensor(e["ends"])
        starts[r, :m] = torch.tensor(e["starts"])
        labels[r, :m] = torch.tensor(e["labels"])
        pos[r, :m] = torch.tensor(e["pos"])
        smask[r, :m] = True
        dom[r] = e["domain"]
    return {k: v.to(device) for k, v in dict(ids=ids, ends=ends, starts=starts, labels=labels,
                                             pos=pos, smask=smask, dom=dom).items()}


def build_examples(docs, l2i, domain2id, rng=None):
    out = []
    for d in docs:
        label_ids = [l2i[x] for x in d["labels"]] if d.get("labels") is not None else None
        for w in make_windows(d["sent_ids"], rng):
            out.append(window_example(d, label_ids, w, domain2id[d["domain"]]))
    return out


# ============================================================================
# 5. DATA REPORT  (everything later steps need, in one json)
# ============================================================================
def compute_class_weights(counts):
    """counts: (n_domains, K) train label counts -> (n_domains, K) capped inverse-sqrt weights.
    Labels with ZERO examples in a domain get weight 0 there (they can never be a target in that domain)."""
    D, K = counts.shape
    pooled = counts.sum(0)
    W = np.zeros((D, K), dtype=np.float32)
    for d in range(D):
        c = counts[d] if PER_DOMAIN_WEIGHTS else pooled
        present = c > 0
        N, Kd = c.sum(), present.sum()
        W[d, present] = np.clip((N / (Kd * c[present])) ** CLASS_WEIGHT_POWER, *CLASS_WEIGHT_CLIP)
    return W


def rare_mask_from_counts(counts):
    """(D, K) bool: label is present in the domain's train set but below RARE_FRAC of its sentences"""
    share = counts / np.maximum(counts.sum(1, keepdims=True), 1)
    return (counts > 0) & (share < RARE_FRAC)


def data_report(data, labels, l2i, tok):
    rep = {"labels": labels, "num_labels": len(labels), "splits": {}}
    for name, docs in data.items():
        tokenize_documents(docs, tok)
        sent_tok = np.array([len(s) for d in docs for s in d["sent_ids"]])
        raw_tok = [len(tok.encode(SENT_PREFIX + s.strip())) for d in docs for s in d["text"]]
        n_sent = [len(d["text"]) for d in docs]
        entry = {
            "documents": len(docs), "sentences": int(sum(n_sent)),
            "sentences_per_doc": {"mean": float(np.mean(n_sent)), "median": float(np.median(n_sent)),
                                  "max": int(max(n_sent))},
            "tokens_per_sentence": {"mean": float(np.mean(raw_tok)), "p95": float(np.percentile(raw_tok, 95)),
                                    "p99": float(np.percentile(raw_tok, 99)), "max": int(max(raw_tok))},
            f"sentences_truncated_at_{MAX_SENT_TOK}_tokens": int((np.array(raw_tok) > MAX_SENT_TOK).sum()),
            "tokens_per_doc_mean": float(sent_tok.sum() / len(docs)),
            "windows_per_epoch_inference_style": int(sum(len(make_windows(d["sent_ids"])) for d in docs)),
        }
        if name.endswith("_train") or name.endswith("_dev"):
            cnt = Counter(x for d in docs for x in d["labels"])
            entry["label_counts"] = {l: cnt.get(l, 0) for l in labels}
        rep["splits"][name] = entry

    # per-domain label counts from the TRAIN splits (one shared model, but weights/priors are domain specific)
    counts = np.zeros((len(DOMAINS), len(labels)), dtype=np.int64)
    for di, dom in enumerate(DOMAINS):
        c = Counter(x for d in data[f"{dom}_train"] for x in d["labels"])
        counts[di] = [c.get(l, 0) for l in labels]
    W = compute_class_weights(counts)
    rep["train_label_counts"] = {l: int(counts[:, i].sum()) for i, l in enumerate(labels)}
    rep["train_label_counts_by_domain"] = {dom: {l: int(counts[di, i]) for i, l in enumerate(labels)}
                                           for di, dom in enumerate(DOMAINS)}
    rep["class_weights_by_domain"] = {dom: {l: float(W[di, i]) for i, l in enumerate(labels)}
                                      for di, dom in enumerate(DOMAINS)}
    rep["labels_absent_from_train_by_domain"] = {dom: [l for i, l in enumerate(labels) if counts[di, i] == 0]
                                                 for di, dom in enumerate(DOMAINS)}
    rep["window_note"] = (f"MAX_LEN={MAX_LEN}, TARGET_TOK={TARGET_TOK}, MAX_SENT_TOK={MAX_SENT_TOK}")

    log("=" * 90)
    log("DATA REPORT")
    log("=" * 90)
    for name, e in rep["splits"].items():
        log(f"{name:9s} docs={e['documents']:3d} sents={e['sentences']:6d} "
            f"tok/sent mean={e['tokens_per_sentence']['mean']:.1f} p99={e['tokens_per_sentence']['p99']:.0f} "
            f"max={e['tokens_per_sentence']['max']} | truncated={e[f'sentences_truncated_at_{MAX_SENT_TOK}_tokens']} "
            f"| windows/epoch={e['windows_per_epoch_inference_style']}")
    rare = rare_mask_from_counts(counts)
    log("\ntrain label counts and class weights per domain  (R = rare -> windows oversampled):")
    log(f"  {'':24s} " + " | ".join(f"{dom:>6s} count  share  weight" for dom in DOMAINS))
    for i, l in enumerate(labels):
        cells = []
        for di in range(len(DOMAINS)):
            share = 100 * counts[di, i] / max(counts[di].sum(), 1)
            cells.append(f"{counts[di, i]:12d} {share:5.2f}% {W[di, i]:6.2f}{'R' if rare[di, i] else ' '}")
        log(f"  {l:24s} " + " | ".join(cells))
    for di, dom in enumerate(DOMAINS):
        if rep["labels_absent_from_train_by_domain"][dom]:
            log(f"labels with ZERO {dom} train examples:", rep["labels_absent_from_train_by_domain"][dom])
        dev_c = Counter(x for d in data[f"{dom}_dev"] for x in d["labels"])
        missed = {l: n for l, n in dev_c.items() if counts[di, l2i[l]] == 0}
        if missed:
            log(f"WARNING: {dom}_dev contains labels unseen in {dom}_train (cannot be learned in-domain): {missed}")
    return rep, torch.tensor(W, dtype=torch.float32), counts


# ============================================================================
# 6. RR MODEL = Model 2 (+LoRMA)  +  sentence head
# ============================================================================
class RRModel(nn.Module):
    def __init__(self, lm: DecoderLM, n_labels, n_domains=2):
        super().__init__()
        self.lm = lm
        d = lm.cfg.d_model
        d_in = d * (2 if POOLING == "last_mean" else 1)
        self.pos_emb = nn.Embedding(N_POS_BUCKETS, 16) if USE_POS_FEATURE else None
        self.dom_emb = nn.Embedding(n_domains, 8) if USE_DOMAIN_FEATURE else None
        d_in += (16 if USE_POS_FEATURE else 0) + (8 if USE_DOMAIN_FEATURE else 0)
        self.head = nn.Sequential(nn.Dropout(HEAD_DROPOUT), nn.Linear(d_in, HEAD_HIDDEN), nn.GELU(),
                                  nn.Dropout(HEAD_DROPOUT), nn.Linear(HEAD_HIDDEN, n_labels))

    def head_parameters(self):
        mods = [self.head] + [m for m in (self.pos_emb, self.dom_emb) if m is not None]
        return [p for m in mods for p in m.parameters()]

    def forward(self, b):
        h = self.lm.hidden(b["ids"])                                         # (B, T, d)
        B, S = b["ends"].shape
        d = h.size(-1)
        last = h.gather(1, b["ends"][:, :, None].expand(B, S, d))             # (B, S, d)
        feats = [last]
        if POOLING == "last_mean":
            csum = torch.cat([torch.zeros(B, 1, d, device=h.device, dtype=h.dtype), h.cumsum(1)], dim=1)
            hi = csum.gather(1, (b["ends"] + 1)[:, :, None].expand(B, S, d))
            lo = csum.gather(1, b["starts"][:, :, None].expand(B, S, d))
            cnt = (b["ends"] + 1 - b["starts"]).clamp(min=1)[:, :, None].to(h.dtype)
            feats.append((hi - lo) / cnt)
        if self.pos_emb is not None:
            feats.append(self.pos_emb(b["pos"]).to(h.dtype))
        if self.dom_emb is not None:
            feats.append(self.dom_emb(b["dom"])[:, None, :].expand(B, S, -1).to(h.dtype))
        return self.head(torch.cat(feats, dim=-1))                            # (B, S, n_labels)


# ---- adapter-only checkpoints (a few MB instead of the full 100M+ model) ----
def trainable_state(model):
    keep = {n for n, p in model.named_parameters() if p.requires_grad}
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if n in keep}


def save_adapter(model, path, extra):
    torch.save({"state": trainable_state(model), "adapter": f"lorma-{LORMA_VARIANT}",
                "lorma": dict(variant=LORMA_VARIANT, r=LORMA_R, alpha=LORMA_ALPHA, dropout=LORMA_DROPOUT,
                              targets=list(LORMA_TARGETS)),
                "head": dict(pooling=POOLING, pos=USE_POS_FEATURE, dom=USE_DOMAIN_FEATURE), **extra}, path + ".tmp")
    os.replace(path + ".tmp", path)


def load_adapter(model, path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    assert ck.get("lorma", {}).get("variant") == LORMA_VARIANT, \
        f"checkpoint holds {ck.get('adapter')}, config is lorma-{LORMA_VARIANT}"
    missing = model.load_state_dict(ck["state"], strict=False)
    assert not missing.unexpected_keys, missing.unexpected_keys
    return ck


# ---- find + load the Model 2 pretrained weights ----
def walk_input():
    if not os.path.isdir(INPUT_ROOT):
        return []
    return list(os.walk(INPUT_ROOT))


def find_model2_ckpt():
    if MODEL2_CKPT_PATH:
        return MODEL2_CKPT_PATH
    best = None                                                    # (val_loss, path): lowest-val-loss best.pt of this tag
    for d, _, files in walk_input():
        if "best.pt" in files and "best_meta.json" in files:
            try:
                meta = read_json(os.path.join(d, "best_meta.json"))
            except Exception:
                continue
            if meta.get("tag") == MODEL2_TAG and (best is None or meta["val_loss"] < best[0]):
                best = (meta["val_loss"], os.path.join(d, "best.pt"))
    if best is not None:
        return best[1]
    newest = None                                                  # (global_step, path): newest latest.pt of this tag
    for d, _, files in walk_input():
        if "latest.pt" in files and "latest_meta.json" in files:
            try:
                meta = read_json(os.path.join(d, "latest_meta.json"))
            except Exception:
                continue
            if meta.get("tag") == MODEL2_TAG and (newest is None or meta["global_step"] > newest[0]):
                newest = (meta["global_step"], os.path.join(d, "latest.pt"))
    if newest is not None:
        return newest[1]
    for d, _, files in walk_input():                               # last resort: a bare best.pt without its meta file
        if "best.pt" in files:
            log(f"WARNING: using {d}/best.pt without a best_meta.json - the tag inside the file is checked after loading")
            return os.path.join(d, "best.pt")
    raise FileNotFoundError("Model 2 checkpoint not found - attach the pretraining output (best.pt + best_meta.json)")


def build_base_model(tok, device):
    path = find_model2_ckpt()
    ck = torch.load(path, map_location="cpu", weights_only=False)
    if ck.get("tag") not in (None, MODEL2_TAG):
        raise ValueError(f"{path} belongs to another model (tag={ck.get('tag')!r}, expected {MODEL2_TAG!r})")
    cfg_dict = dict(ck["config"])
    cfg_dict["dropout"] = FT_DROPOUT                  # dropout has no parameters, so the weights still load
    cfg = ModelConfig(**cfg_dict)
    assert cfg.vocab_size == tok.vocab_size, (cfg.vocab_size, tok.vocab_size)
    lm = DecoderLM(cfg)
    lm.load_state_dict(ck["model"], strict=True)
    log(f"loaded Model 2 from {path} | {sum(p.numel() for p in lm.parameters()) / 1e6:.1f}M params | cfg={cfg}")
    del ck
    return lm.to(device)


# ============================================================================
# 7. TRAIN / EVAL / PREDICT
# ============================================================================
def macro_prf(y_true, y_pred, labels):
    """macro P/R/F1 over labels that occur in gold OR prediction (sklearn default); also per-class table"""
    from sklearn.metrics import precision_recall_fscore_support
    present = sorted(set(y_true) | set(y_pred))
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=present, average=None, zero_division=0)
    per = {labels[c]: dict(p=float(p[i]), r=float(r[i]), f1=float(f[i]), support=int(s[i])) for i, c in enumerate(present)}
    return dict(macro_p=float(np.mean(p)), macro_r=float(np.mean(r)), macro_f1=float(np.mean(f)), per_class=per)


@torch.no_grad()
def predict_docs(model, docs, domain2id, pad_id, device, eval_bs=8, return_logits=False):
    """-> {doc_id: [label_id per sentence]} (and optionally {doc_id: (n, K) logits})"""
    model.eval()
    preds, logit_store = {}, {}
    for d in docs:
        n = len(d["sent_ids"])
        exs = [window_example(d, None, w, domain2id[d["domain"]]) for w in make_windows(d["sent_ids"])]
        out = np.full(n, -1, dtype=np.int64)
        lg = None
        for i in range(0, len(exs), eval_bs):
            chunk = exs[i:i + eval_bs]
            b = collate(chunk, pad_id, device)
            with amp_ctx():
                logits = model(b).float()
            if lg is None:
                lg = np.zeros((n, logits.size(-1)), dtype=np.float32)
            for r, e in enumerate(chunk):
                m = len(e["sent_idx"])
                lr = logits[r, :m].cpu().numpy()
                lg[e["sent_idx"]] = lr
                out[e["sent_idx"]] = lr.argmax(-1)
        assert (out >= 0).all() and len(out) == n, f"{d['id']}: not every sentence was predicted"
        preds[d["id"]] = out.tolist()
        logit_store[d["id"]] = lg
    return (preds, logit_store) if return_logits else preds


def evaluate_dev(model, dev_docs, labels, l2i, domain2id, pad_id, device):
    preds = predict_docs(model, dev_docs, domain2id, pad_id, device)
    res = {}
    for name, sel in (("ALL", dev_docs), ("CL", [d for d in dev_docs if d["domain"] == "CL"]),
                      ("IT", [d for d in dev_docs if d["domain"] == "IT"])):
        if not sel:
            continue
        yt = [l2i[x] for d in sel for x in d["labels"]]
        yp = [p for d in sel for p in preds[d["id"]]]
        res[name] = macro_prf(yt, yp, labels)
        res[name]["acc"] = float(np.mean(np.array(yt) == np.array(yp)))
    return res


def weighted_ce(logits, b, class_w):
    """class_w: (n_domains, K).  Each target token is weighted by ITS domain's weight for ITS label;
    normalised by the summed weight of the micro-batch (a weighted mean, so the loss scale stays ~constant)."""
    K = logits.size(-1)
    y = b["labels"]
    valid = y >= 0
    ce = F.cross_entropy(logits.float().reshape(-1, K), y.reshape(-1), ignore_index=-100,
                         reduction="none", label_smoothing=LABEL_SMOOTHING).view_as(y)
    w = class_w[b["dom"][:, None].expand_as(y), y.clamp(min=0)] * valid
    return (ce * w).sum() / w.sum().clamp(min=1e-6)


def oversample_rare(exs, rare_mask):
    out = []
    for e in exs:
        out.append(e)
        if any(rare_mask[e["domain"], y] for y in e["labels"] if y >= 0):
            out.extend([e] * (RARE_REPEAT - 1))
    return out


def make_epoch_examples(train_docs, l2i, domain2id, rng, rare_mask):
    exs = build_examples(train_docs, l2i, domain2id, rng)      # fresh random window boundaries
    return oversample_rare(exs, rare_mask) if OVERSAMPLE_RARE else exs


def build_post(counts):
    """per-domain log-priors (for logit adjustment) and the set of labels seen in each domain's train split"""
    allowed = counts > 0
    prior = counts / np.maximum(counts.sum(1, keepdims=True), 1)
    logp = np.log(np.where(allowed, prior, 1.0))
    for d in range(len(counts)):                    # unseen label: no boost (treated like the most frequent label)
        logp[d, ~allowed[d]] = logp[d, allowed[d]].max()
    return {"log_prior": logp.astype(np.float32), "allowed": allowed, "tau": 0.0, "use_mask": False}


def decode_logits(logit_store, docs, domain2id, post, tau=None, use_mask=None):
    """argmax of  logits - tau * log(prior_domain)   (tau=0: plain argmax), optionally forbidding labels unseen in-domain"""
    tau = post["tau"] if tau is None else tau
    use_mask = post["use_mask"] if use_mask is None else use_mask
    preds = {}
    for d in docs:
        di = domain2id[d["domain"]]
        lg = logit_store[d["id"]].copy()
        if tau:
            lg -= tau * post["log_prior"][di][None, :]
        if use_mask:
            lg[:, ~post["allowed"][di]] = -1e9
        preds[d["id"]] = lg.argmax(-1).tolist()
    return preds


def calibrate_postproc(model, dev_docs, labels, l2i, domain2id, pad_id, device, counts):
    """Two cheap knobs tuned on dev macro-F1: tau (prior correction strength) and the unseen-label mask.
    Only 2 degrees of freedom, so it cannot overfit the small dev set the way per-class biases would."""
    _, store = predict_docs(model, dev_docs, domain2id, pad_id, device, return_logits=True)
    post = build_post(counts)
    yt = [l2i[x] for d in dev_docs for x in d["labels"]]
    best = None
    for use_mask in (False, True):
        for tau in TAU_GRID:
            pr = decode_logits(store, dev_docs, domain2id, post, tau, use_mask)
            yp = [p for d in dev_docs for p in pr[d["id"]]]
            f1 = macro_prf(yt, yp, labels)["macro_f1"]
            log(f"  calibration: tau={tau:.2f} mask_unseen={use_mask!s:5s} -> dev macro-F1 {f1:.4f}")
            if best is None or f1 > best[0] + 1e-4:          # ties go to the simpler setting (tried first)
                best = (f1, tau, use_mask)
    post["tau"], post["use_mask"] = best[1], best[2]
    log(f"calibration chosen: tau={best[1]} mask_unseen={best[2]} (dev macro-F1 {best[0]:.4f})")
    return post, best[0]


def lr_scale(step, total, warmup):
    if step < warmup:
        return (step + 1) / max(1, warmup)
    prog = (step - warmup) / max(1, total - warmup)
    return 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))


def train(model, train_docs, dev_docs, labels, l2i, domain2id, class_w, tok, device, rare_mask):
    os.makedirs(CKPT_DIR, exist_ok=True)
    K = len(labels)
    adapter_params = [p for n, p in model.lm.named_parameters() if p.requires_grad]
    head_params = model.head_parameters()
    n_lora, n_head = sum(p.numel() for p in adapter_params), sum(p.numel() for p in head_params)
    n_all = sum(p.numel() for p in model.lm.parameters())
    log(f"trainable: LoRMA-{LORMA_VARIANT}(+norm) {n_lora / 1e6:.2f}M + head {n_head / 1e6:.2f}M "
        f"= {100 * (n_lora + n_head) / (n_all + n_head):.2f}% of all parameters")

    opt = torch.optim.AdamW(
        [{"params": adapter_params, "lr": LR_LORMA, "base_lr": LR_LORMA, "weight_decay": 0.0},
         {"params": head_params, "lr": LR_HEAD, "base_lr": LR_HEAD, "weight_decay": WEIGHT_DECAY}],
        betas=(0.9, 0.999), eps=1e-8)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP["enabled"] and AMP["dtype"] == torch.float16)
    class_w = class_w.to(device)

    rng = random.Random(SEED)
    steps_per_epoch = max(1, math.ceil(len(make_epoch_examples(train_docs, l2i, domain2id, random.Random(SEED), rare_mask)) / (MICRO_BS * ACCUM)))
    total_steps, warmup = EPOCHS * steps_per_epoch, int(WARMUP_FRAC * EPOCHS * steps_per_epoch)
    log(f"epochs={EPOCHS} steps/epoch~{steps_per_epoch} total~{total_steps} "
        f"effective batch={MICRO_BS * ACCUM} windows")

    best_f1, best_epoch, bad, gstep = -1.0, -1, 0, 0
    log_path = os.path.join(CKPT_DIR, "train_log.csv")
    with open(log_path, "w", newline="") as f:
        csv.writer(f).writerow(["epoch", "train_loss", "dev_macro_f1", "dev_cl_f1", "dev_it_f1", "dev_acc", "sec"])

    for epoch in range(EPOCHS):
        t0 = time.time()
        model.train()
        exs = make_epoch_examples(train_docs, l2i, domain2id, rng, rare_mask)   # fresh windows + rare-label oversampling
        rng.shuffle(exs)
        run, nrun = 0.0, 0
        for i in range(0, len(exs), MICRO_BS * ACCUM):
            group = exs[i:i + MICRO_BS * ACCUM]
            n_micro = math.ceil(len(group) / MICRO_BS)
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * lr_scale(gstep, total_steps, warmup)
            for j in range(0, len(group), MICRO_BS):
                b = collate(group[j:j + MICRO_BS], tok.pad_id, device)
                with amp_ctx():
                    logits = model(b)
                loss = weighted_ce(logits, b, class_w)      # per-domain capped inverse-sqrt weights
                scaler.scale(loss / n_micro).backward()
                run += loss.item()
                nrun += 1
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(adapter_params + head_params, GRAD_CLIP)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            gstep += 1

        res = evaluate_dev(model, dev_docs, labels, l2i, domain2id, tok.pad_id, device)
        f1 = res["ALL"]["macro_f1"]
        row = [epoch + 1, f"{run / max(nrun, 1):.4f}", f"{f1:.4f}", f"{res.get('CL', {}).get('macro_f1', float('nan')):.4f}",
               f"{res.get('IT', {}).get('macro_f1', float('nan')):.4f}", f"{res['ALL']['acc']:.4f}", f"{time.time() - t0:.0f}"]
        with open(log_path, "a", newline="") as f:
            csv.writer(f).writerow(row)
        improved = f1 > best_f1
        log(f"[epoch {epoch + 1:2d}/{EPOCHS}] loss {row[1]} | dev macro-F1 {row[2]} (CL {row[3]} / IT {row[4]}) "
            f"acc {row[5]} | {row[6]}s {'<-- best' if improved else ''}")
        if improved:
            best_f1, best_epoch, bad = f1, epoch + 1, 0
            save_adapter(model, os.path.join(CKPT_DIR, "best_adapter.pt"), {"epoch": epoch + 1, "dev_macro_f1": f1})
        else:
            bad += 1
            if bad >= PATIENCE:
                log(f"early stop: no dev improvement for {PATIENCE} epochs")
                break
    log(f"best dev macro-F1 {best_f1:.4f} at epoch {best_epoch}")
    return best_f1


def write_submission(preds_by_id, data, labels):
    expected = {d["id"]: len(d["text"]) for k in ("CL_test", "IT_test") for d in data[k]}
    assert set(preds_by_id) == set(expected), "prediction ids differ from the test ids"
    out, valid = {}, set(labels)
    for doc_id, n in expected.items():                       # test-file order, CL then IT
        lab = [labels[i] for i in preds_by_id[doc_id]]
        assert len(lab) == n and all(l in valid for l in lab), doc_id
        out[doc_id] = lab
    os.makedirs(WORK_DIR, exist_ok=True)
    pred_path, zip_path = os.path.join(WORK_DIR, "prediction.json"), os.path.join(WORK_DIR, "submission.zip")
    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(pred_path, arcname="prediction.json")
    dist = Counter(l for v in out.values() for l in v)
    log(f"wrote {pred_path} and {zip_path} | {len(out)} docs, {sum(len(v) for v in out.values())} sentences")
    log("predicted label distribution:", dict(dist.most_common()))


# ============================================================================
# 8. MAIN
# ============================================================================
def run_self_checks(model, device):
    """pi rotation = explicit loop, M = I at init (adapted == base), factorised forward == merged weights,
    unmerge restores W0 exactly, and only adapters + head may train"""
    model.eval()
    X = torch.randn(7, 7)                                              # rotation vs an explicit torch.roll loop
    assert torch.equal(pi_rotate(X), torch.stack([torch.roll(X[i], shifts=i) for i in range(7)])), "pi_rotate is wrong"
    one_e0 = torch.zeros(7, 7)
    one_e0[:, 0] = 1.0                                                 # 1 e_0^T  ->  identity
    assert torch.equal(pi_rotate(one_e0), torch.eye(7)), "I_pi(1 e_0^T) is not the identity"

    ids = torch.randint(3, model.lm.cfg.vocab_size, (1, 64), device=device)
    mods = lorma_modules(model)
    saved_B = [m.lorma_B.detach().clone() for m in mods]
    saved, AMP["enabled"] = AMP["enabled"], False
    try:
        with torch.no_grad():
            set_lorma_enabled(model, False)
            base = model.lm.hidden(ids)                                # the untouched pretrained model
            set_lorma_enabled(model, True)
            init = model.lm.hidden(ids)                                # adapters at their initial values
            for m in mods:                                             # make M != I so merge is actually tested
                m.lorma_B.add_(torch.randn_like(m.lorma_B) * 0.01)
            adapted = model.lm.hidden(ids)
            w_before = [m.base.weight.detach().clone() for m in mods]
            merge_lorma(model)
            merged = model.lm.hidden(ids)
            unmerge_lorma(model)
            back = model.lm.hidden(ids)
            w_restored = all(torch.equal(m.base.weight, w) for m, w in zip(mods, w_before))
        assert (init - base).abs().max() < 1e-5, "LoRMA is not the identity at init"
        assert (adapted - init).abs().max() > 1e-6, "LoRMA had no effect"
        assert (adapted - merged).abs().max() < 1e-3, "merged LoRMA differs from adapted forward"
        assert w_restored and (adapted - back).abs().max() < 1e-4, "unmerge is not lossless"
    finally:
        with torch.no_grad():
            for m, b in zip(mods, saved_B):
                m.lorma_B.copy_(b)
        set_lorma_enabled(model, True)
        AMP["enabled"] = saved
    frozen_bad = [n for n, p in model.lm.named_parameters()
                  if p.requires_grad and "lorma_" not in n and not n.startswith("final_norm")]
    assert not frozen_bad, f"base weights are trainable: {frozen_bad[:3]}"
    log(f"self-checks passed (LoRMA-{LORMA_VARIANT}): M = I at init, merge == adapted forward, unmerge is lossless, "
        f"base weights are frozen")


def main():
    set_seed()
    os.makedirs(WORK_DIR, exist_ok=True)
    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")
    if use_cuda:
        AMP["enabled"] = True
        AMP["dtype"] = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
        log(f"GPU: {torch.cuda.get_device_name()} | autocast dtype: {AMP['dtype']}")
    else:
        log("WARNING: no GPU visible - training will be very slow")

    data, labels, l2i = load_phase3()
    tok = load_tokenizer()
    domain2id = {d: i for i, d in enumerate(DOMAINS)}

    report, class_w, counts = data_report(data, labels, l2i, tok)
    rare_mask = rare_mask_from_counts(counts)
    with open(os.path.join(WORK_DIR, "data_report.json"), "w") as f:
        json.dump(report, f, indent=2)

    lm = build_base_model(tok, device)
    n_wrapped = inject_lorma(lm)
    log(f"LoRMA-{LORMA_VARIANT} injected into {n_wrapped} linear layers (r={LORMA_R}, alpha={LORMA_ALPHA} [plus only], targets={LORMA_TARGETS})")
    model = RRModel(lm, len(labels)).to(device)
    run_self_checks(model, device)

    train_docs = data["CL_train"] + data["IT_train"]
    dev_docs = data["CL_dev"] + data["IT_dev"]
    train(model, train_docs, dev_docs, labels, l2i, domain2id, class_w, tok, device, rare_mask)

    # final dev report with the best adapter
    ck = load_adapter(model, os.path.join(CKPT_DIR, "best_adapter.pt"))
    res = evaluate_dev(model, dev_docs, labels, l2i, domain2id, tok.pad_id, device)
    log(f"\nBEST adapter (epoch {ck['epoch']}) on dev: macro-P {res['ALL']['macro_p']:.4f} "
        f"macro-R {res['ALL']['macro_r']:.4f} macro-F1 {res['ALL']['macro_f1']:.4f}")
    for l, v in res["ALL"]["per_class"].items():
        log(f"  {l:24s} P {v['p']:.3f} R {v['r']:.3f} F1 {v['f1']:.3f} support {v['support']}")
    with open(os.path.join(WORK_DIR, "dev_report.json"), "w") as f:
        json.dump(res, f, indent=2)

    post = build_post(counts)
    if POSTHOC_CALIBRATION:
        log("\npost-hoc calibration on dev:")
        post, cal_f1 = calibrate_postproc(model, dev_docs, labels, l2i, domain2id, tok.pad_id, device, counts)
        _, dstore = predict_docs(model, dev_docs, domain2id, tok.pad_id, device, return_logits=True)
        dpred = decode_logits(dstore, dev_docs, domain2id, post)
        for name in ("ALL",) + tuple(DOMAINS):
            sel = dev_docs if name == "ALL" else [d for d in dev_docs if d["domain"] == name]
            r = macro_prf([l2i[x] for d in sel for x in d["labels"]], [p for d in sel for p in dpred[d["id"]]], labels)
            log(f"  calibrated dev {name:3s}: macro-P {r['macro_p']:.4f} macro-R {r['macro_r']:.4f} macro-F1 {r['macro_f1']:.4f}")
        with open(os.path.join(WORK_DIR, "postproc.json"), "w") as f:
            json.dump({"tau": post["tau"], "mask_unseen": bool(post["use_mask"]), "dev_macro_f1": cal_f1}, f, indent=2)

    if RUN_PREDICTION:
        test_docs = data["CL_test"] + data["IT_test"]
        tokenize_documents(test_docs, tok)
        merge_lorma(model)                        # fold M into the weights: no extra inference cost
        _, tstore = predict_docs(model, test_docs, domain2id, tok.pad_id, device, return_logits=True)
        preds = decode_logits(tstore, test_docs, domain2id, post)
        write_submission(preds, data, labels)
    log("done.")


if __name__ == "__main__":
    main()